In [1]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import math
# Tải dữ liệu
from ucimlrepo import fetch_ucirepo
phishing_websites = fetch_ucirepo(id=327)
X = phishing_websites.data.features
y = phishing_websites.data.targets

# Chia tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y['result'], test_size=0.2, random_state=42, stratify=y)

In [2]:
y_train

1758    -1
10910   -1
7412     1
5805     1
9259     1
        ..
6980     1
10730   -1
7739    -1
7082    -1
9945     1
Name: result, Length: 8844, dtype: int64

In [3]:
from sklearn.model_selection import StratifiedKFold


In [63]:
class Hyperband:
    def __init__(self, estimator, param_distributions, max_iter=81, eta=3, random_state=None):
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.max_iter = max_iter  # Số lần lặp tối đa
        self.eta = eta  # Hệ số giảm
        self.random_state = random_state
        self.s_max = int(np.log(self.max_iter) / np.log(self.eta))
        self.B = (self.s_max + 1) * self.max_iter
        self.cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        self.T = []
        if self.random_state is not None:
            np.random.seed(self.random_state)
    
    def sample_params(self):
        """
        Lấy mẫu tham số từ các phân phối hoặc danh sách giá trị.
        Hỗ trợ:
            - scipy.stats distributions (randint, uniform,...)
            - list giá trị rời rạc
        """
        sampled_params = {}
        for param, dist in self.param_distributions.items():
            sampled_params[param] = dist.rvs()
        return sampled_params
    
    def fit(self, X, y):
        """Thực hiện tối ưu hóa siêu tham số với Hyperband"""
        best_score = -np.inf
        best_params = None
        results = []
        total_runs = 0
        for s in reversed(range(self.s_max +1)):
            print(s)
            n = math.ceil(self.B / self.max_iter * self.eta ** s / (s + 1))
            r = self.max_iter * self.eta ** (-s)
            T = [self.sample_params() for _ in range(n)]     
            for i in range(s + 1):
                n_i = math.floor(n * self.eta ** (-i))
                r_i = int(r * self.eta ** i)
                print(i,n_i, r_i)
                scores = []
                for t in T:
                    # print("Đang thử nghiệm param", t)
                    params = t.copy()
                    result = {}
                    params['n_estimators'] = r_i
                    print(t)
                    self.estimator.set_params(**params)
                    result = {'params': params}
                    total_runs += 1
                    score = cross_val_score(estimator=self.estimator, X=X, y=y, cv=self.cv, scoring='accuracy').mean()
                    scores.append(score)
                    print(result['params'])
                    result['score'] = score
                    result['s'] = s
                    result['i'] = i
                    result['n_i'] = n_i
                    result['r_i'] = r_i
                    results.append(result)
                k = math.floor(n_i / n)
                top_k_indices = np.argsort(scores)[-k:][::-1]
                print(top_k_indices)
                
        for result in results:
            if result['score'] > best_score:
                best_score = result['score']
                best_params = result['params']
        self.best_params_ = best_params
        self.best_score_ = best_score
        self.total_runs = total_runs
        self.results = results
        return self

In [65]:
from scipy.stats import randint, uniform

# Định nghĩa không gian tham số rộng hơn cho LightGBM
param_distributions = {
    'num_leaves': randint(20, 100),  # Số lá trong cây
    'max_depth': randint(3, 12),     # Độ sâu tối đa
    'learning_rate': uniform(0.01, 0.3),  # Tốc độ học
    'min_child_samples': randint(10, 50),  # Số mẫu tối thiểu trong mỗi lá
    'subsample': uniform(0.6, 0.4),   # Tỷ lệ mẫu sử dụng cho mỗi cây
    'colsample_bytree': uniform(0.6, 0.4),  # Tỷ lệ features sử dụng cho mỗi cây
    'reg_alpha': uniform(0, 1),       # L1 regularization
    'reg_lambda': uniform(0, 1),      # L2 regularization
    'min_child_weight': uniform(0, 1)  # Trọng số tối thiểu cho mỗi lá
}

In [66]:
from lightgbm import LGBMClassifier

In [67]:
hb = Hyperband(
    estimator=LGBMClassifier(random_state=42, n_jobs=1),
    param_distributions=param_distributions,
    max_iter=81,
    eta=3,
    random_state=42
)

hb.fit(X_train, y_train)

print("Best parameters:", hb.best_params_)
print("Best score:", hb.best_score_)

4
0 81 1
{'num_leaves': 71, 'max_depth': 10, 'learning_rate': 0.189597545259111, 'min_child_samples': 48, 'subsample': 0.7783331011414365, 'colsample_bytree': 0.6399899663272012, 'reg_alpha': 0.45924889196586716, 'reg_lambda': 0.33370861113902184, 'min_child_weight': 0.14286681792194078, 'n_estimators': 1}
[LightGBM] [Info] Number of positive: 3941, number of negative: 3134
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001034 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 89
[LightGBM] [Info] Number of data points in the train set: 7075, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.557032 -> initscore=0.229124
[LightGBM] [Info] Start training from score 0.229124
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3941, number of negative: 31

In [68]:
hb.results

[{'params': {'num_leaves': 71,
   'max_depth': 10,
   'learning_rate': 0.189597545259111,
   'min_child_samples': 48,
   'subsample': 0.7783331011414365,
   'colsample_bytree': 0.6399899663272012,
   'reg_alpha': 0.45924889196586716,
   'reg_lambda': 0.33370861113902184,
   'min_child_weight': 0.14286681792194078,
   'n_estimators': 81},
  'score': 0.8934863626713458,
  's': 4,
  'i': 0,
  'n_i': 81,
  'r_i': 1},
 {'params': {'num_leaves': 22,
   'max_depth': 8,
   'learning_rate': 0.02692347370813008,
   'min_child_samples': 33,
   'subsample': 0.9329770563201687,
   'colsample_bytree': 0.6849356442713105,
   'reg_alpha': 0.18182496720710062,
   'reg_lambda': 0.18340450985343382,
   'min_child_weight': 0.3042422429595377,
   'n_estimators': 81},
  'score': 0.5569878040358205,
  's': 4,
  'i': 0,
  'n_i': 81,
  'r_i': 1},
 {'params': {'num_leaves': 41,
   'max_depth': 11,
   'learning_rate': 0.09736874205941257,
   'min_child_samples': 37,
   'subsample': 0.9895022075365837,
   'colsam

In [69]:
params = [r['params'] for r in hb.results]
score = [r['score'] for r in hb.results]
i = [r['i'] for r  in hb.results]
s = [r['s'] for r in hb.results]
r_i = [r['r_i'] for r in hb.results]
df = pd.DataFrame(params)


In [70]:
df['score'] = score
df['s'] = s
df['i'] = i
df['r_i'] = r_i

In [53]:
df['n_estimators'].unique()

array([81], dtype=int64)

In [71]:
df.to_csv('D:\output.csv')

<>:1: SyntaxWarning: invalid escape sequence '\o'
<>:1: SyntaxWarning: invalid escape sequence '\o'
C:\Users\duclh5\AppData\Local\Temp\ipykernel_29156\4021180612.py:1: SyntaxWarning: invalid escape sequence '\o'
  df.to_csv('D:\output.csv')


In [27]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
estimator = LGBMClassifier(random_state=42)
params = hb.results[145]['params']
estimator.set_params(**params)

LGBMClassifier(colsample_bytree=0.92797876696324,
               learning_rate=0.018774184910367746, max_depth=11,
               min_child_samples=32, min_child_weight=0.001120111480109487,
               n_estimators=27, num_leaves=64, random_state=42,
               reg_alpha=0.22862516152492351, reg_lambda=0.9087013092693679,
               subsample=0.6020580683018859)

In [ ]:
scores = []
for i in range(5):
    score = cross_val_score(estimator=estimator, X=X_train, y=y_train, cv=cv, scoring='accuracy').mean()
    scores.append(score)

[LightGBM] [Info] Number of positive: 3941, number of negative: 3134
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 89
[LightGBM] [Info] Number of data points in the train set: 7075, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.557032 -> initscore=0.229124
[LightGBM] [Info] Start training from score 0.229124
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [32]:
scores

[0.9348707887729601,
 0.9348707887729601,
 0.9348707887729601,
 0.9348707887729601,
 0.9348707887729601]